In [1]:
include("../RayTracing.jl")

Main.RayTracing

In [2]:
parsed_args = RayTracing.parse_commandline()
parsed_args["scene-number"] = 4
    
# set up logging
logger = RayTracing.setup_logging(parsed_args["debug"])
RayTracing.global_logger(logger)

# set random seed
RayTracing.Random.seed!(parsed_args["seed"])

DIM = 25
parsed_args["image-dim"] = DIM

25

In [3]:
I, scene = RayTracing.build_scene(parsed_args)


There are 37 objects in the scene, building BVH
  0.067996 seconds (105.61 k allocations: 7.075 MiB, 99.84% compilation time)
Done building BVH
Using 5 samples per pixel
There are 2 lights in the scene


(Main.RayTracing.BDPTIntegrator(Main.RayTracing.PerspectiveCamera(Main.RayTracing.ProjectiveCamera(Main.RayTracing.CameraCore(Main.RayTracing.Transformation([1.0 0.0 0.0 278.0; 0.0 1.0 0.0 278.0; 0.0 0.0 1.0 -800.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 -278.0; 0.0 1.0 0.0 -278.0; 0.0 0.0 1.0 800.0; 0.0 0.0 0.0 1.0]), 0.0, 1.0, Main.RayTracing.Film([25.0, 25.0], Main.RayTracing.Bounds2([0.0, 0.0], [25.0, 25.0]), 0.001, Main.RayTracing.BoxFilter([0.1, 0.1]), "yeehaw.exr", Main.RayTracing.Pixel[Main.RayTracing.Pixel([0.0, 0.0, 0.0], 0.0, Main.RayTracing.AtomicXYZPBRT(Base.Threads.Atomic{Float64}(0.0), Base.Threads.Atomic{Float64}(0.0), Base.Threads.Atomic{Float64}(0.0))) Main.RayTracing.Pixel([0.0, 0.0, 0.0], 0.0, Main.RayTracing.AtomicXYZPBRT(Base.Threads.Atomic{Float64}(0.0), Base.Threads.Atomic{Float64}(0.0), Base.Threads.Atomic{Float64}(0.0))) … Main.RayTracing.Pixel([0.0, 0.0, 0.0], 0.0, Main.RayTracing.AtomicXYZPBRT(Base.Threads.Atomic{Float64}(0.0), Base.Threads.Atomic{Float64}(0.0), Bas

In [4]:
# Instantiate a Filter
filter = RayTracing.BoxFilter(RayTracing.Pnt2(.1, .1))

image = zeros(Float64, (DIM, DIM))
for x in 0:(DIM-1)
    for y in 0:(DIM-1)
        xmin = x/DIM
        ymin = y/DIM
        xmax = (x+1)/DIM
        ymax = (y+1)/DIM
        parsed_args["crop-window"] = [xmin, ymin, xmax, ymax]
        println("Working on $([xmin, ymin, xmax, ymax])")

        # Instantiate a Film
        film = RayTracing.Film(
            RayTracing.Pnt2(parsed_args["image-dim"], parsed_args["image-dim"]),
            RayTracing.Bounds2(
                RayTracing.Pnt2(parsed_args["crop-window"][1], parsed_args["crop-window"][2]),
                RayTracing.Pnt2(parsed_args["crop-window"][3], parsed_args["crop-window"][4])),
            filter,
            1.0,
            1.0,
            parsed_args["file-name"]
        )

        # Instantiate a Camera
        look_from = RayTracing.Pnt3(278, 278, -800)
        look_at = RayTracing.Pnt3(278, 278, 0)
        up = RayTracing.Vec3(0, 1, 0)
        screen = RayTracing.Bounds2(RayTracing.Pnt2(-1, -1), RayTracing.Pnt2(1, 1))
        C = RayTracing.PerspectiveCamera(RayTracing.LookAt(look_from, look_at, up), screen, 0.0, 1.0, 0.0, 1e6, 40.0, film)

        # Instantiate a Sampler
        # S = ZSobolSampler(parsed_args["samples-per-pixel"], film.full_resolution, Int8(2))
        S = RayTracing.StratifiedSampler(parsed_args["samples-per-pixel"], parsed_args["jitter"])

        # Instantiate an Integrator
        I = RayTracing.BDPTIntegrator(C, S, parsed_args["max-depth"])

        
        muahaha = @elapsed(RayTracing.render(I, scene, parsed_args, (-1, -1)))
        image[y+1,x+1] = muahaha
    end
end

Working on [0.0, 0.0, 0.04, 0.04]
Rendering 1 tiles
Utilizing 1 threads

Working on [0.0, 0.04, 0.04, 0.08]
Rendering 1 tiles
Utilizing 1 threads

Working on [0.0, 0.08, 0.04, 0.12]
Rendering 1 tiles
Utilizing 1 threads

Working on [0.0, 0.12, 0.04, 0.16]
Rendering 1 tiles
Utilizing 1 threads

Working on [0.0, 0.16, 0.04, 0.2]
Rendering 1 tiles
Utilizing 1 threads

Working on [0.0, 0.2, 0.04, 0.24]
Rendering 1 tiles
Utilizing 1 threads

Working on [0.0, 0.24, 0.04, 0.28]
Rendering 1 tiles
Utilizing 1 threads



BoundsError: BoundsError: attempt to access 2×3×3 Array{Float64, 3} at index [3, 1, 1:3]

In [5]:
spectrum_img = RayTracing.spectrum_from_float.((image .- minimum(image))./(maximum(image) - minimum(image)))
newimage = zeros(RayTracing.RGB, DIM, DIM)
for y in 1:DIM
    for x in 1:DIM
        newimage[y,x] = RayTracing.RGB(spectrum_img[y,x][1], spectrum_img[y,x][2], spectrum_img[y,x][3])
    end
end
RayTracing.OpenEXR.save("hahaha.exr", newimage)